# REGEX: Regular Expressions

As expressões regulares começam em 1951, com o matemático Stephen Cole Kleene, que criou uma notação matemática chamada de "Eventos Regulares". Em 1968 se torna popular para encontrar padrões em texto. Ken Thompson fez algumas implementações, que acabaram virando o código grep ("Global search for Regular Expression and Print matching lines"), utilizado até hoje em sistemas UNIX/LINUX. Em 1986, Henry Spencer, da Universidade de Toronto, publicou na Usenet uma biblioteca de regex escrita em C, que acabou incorporada a inúmeros programas. Foi essa biblioteca que o Perl adotou na versão 2, em 1988 — a linguagem tinha sido criada por Larry Wall no ano anterior. É no Perl que os regexes ganham a expressividade que usamos hoje, e é por isso que até hoje se fala em sintaxe "compatível com Perl" (PCRE) nas ferramentas mais diversas.

O regex é a linguagem mais utilizada para processamento de textos em geral. Não só em processamento de linguagem natural, mas para qualquer processamento de textos para códigos.

Vamos estudar a implementação do regex no python, que possue duas bibliotecas a re e a regex. A implementação re é nativa do Python, a regex é uma biblioteca externa. Em geral a regex é mais completa, portanto vamos utilizá-la.

### Vamos buscar uma propriedade de interesse, por exemplo, "Yield Strength".
![Propriedades Mecânicas](mechanical.png)

### Importando o regex

In [ ]:
!pip install regex
import regex as re

### Definindo a sentença

In [ ]:
sentence = "The yield strength of the AlCuNiFeCr alloy reached 1511 MPa at room temperature."

### Realizando uma busca com regex

In [ ]:
property = re.search("[Yy]ield [Ss]trength", sentence) 

In [ ]:
print(property) #Saída do regex.
print(property[0]) #Somente o que foi encontrado.

O código regex encontrou o padrão "yield strength" na sentença.

Mas o que exatamente significa a expressão regular utilizada?

![Explicando o regex](regex1.png)

Esse é um tipo de regex bastante simples.

### Agora vamos buscar por uma unidade.
No caso a unidade do Yield Strength é Pascal, que pode ser tera, giga, mega ou kilo Pascal.<br>
Para encontrar essa unidade usar o regex [TGMKk]Pa, que dará match para TPa, GPa, MPa, KPa ou kPa.<br>
Queremos que exista um número antes da unidade, que é o que buscamos. Para isso adicionamos o regex \d+\s, onde \d+ é qualquer dígito, e \s é um espaço.

In [ ]:
value = re.search("\d+\s[TGMKk]Pa", sentence)
print(value)
print(value[0])

### Exemplo de regex para achar composição química

Vamos começar tentanto achar elementos químicos.

In [ ]:
sentence = "Meus átomos preferidos são Fe, C, H, O e Br."

In [ ]:
atoms = re.findall(r"\b[A-Z][a-z]?\b", sentence)
print(atoms)

Vejam que mudei o comando do regex, agora utilizei o re.findall, pois ele retorna todas ocorrências, não só a primeira.

### Encontrando fórmulas químicas

Vamos retornar a sentença inicial.

In [ ]:
sentence = "The yield strength of the AlCuNiFeCr alloy reached 1511 MPa at room temperature."

Vamos utilizar um regex para encontrarmos a fórmula da liga que está na sentença.

In [ ]:
compositions = re.findall(r"\b[A-Z][a-z]?(?:[A-Z][a-z]?){4,8}\b", sentence)

In [ ]:
print(compositions)

Mas que diabos é isso professor?<br>
Há ótimos sites para explicar códigos regex: https://regex101.com ou https://regexr.com<br>
Além das IAs generativas é claro.

Agora se nossa fórmula química tiver números também?

In [ ]:
sentence = "The yield strength of the Al0.1Cu0.3Ni0.2Fe0.3Cr0.1 alloy reached 1511 MPa at room temperature."

In [ ]:
#Construído com auxílio do ChatGPT.
compositions = re.findall(r"\b[A-Z][a-z]?\d*(?:\.\d+)?(?:[A-Z][a-z]?\d*(?:\.\d+)?){4,8}\b", sentence)

In [ ]:
print(compositions)

### Vamos rodar para vários abstracts agora.

In [ ]:
#!pip install xlrd
import pandas as pd

Ler todos arquivos da Web of Science:

In [ ]:
dados1 = pd.read_excel('savedrecs1.xls')
dados2 = pd.read_excel('savedrecs2.xls')
dados3 = pd.read_excel('savedrecs3.xls')
dados4 = pd.read_excel('savedrecs4.xls')
dados5 = pd.read_excel('savedrecs5.xls')
dados6 = pd.read_excel('savedrecs6.xls')
dados7 = pd.read_excel('savedrecs7.xls')
dados8 = pd.read_excel('savedrecs8.xls')

Concatenar todos arquivos:

In [ ]:
dados_total = pd.concat([dados1, dados2, dados3, dados4, dados5, dados6, dados7, dados8], ignore_index=True)

In [ ]:
print(dados_total['Abstract'])

Utilizar o Spacy para separar em sentenças e fazer um loop para cada sentença de cada abstract.

Nesse loop aplicaremos os regex.

In [ ]:
import spacy
!python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

In [ ]:
i = -1
for abstract in dados_total['Abstract']:
    #print('Abstract', i)
    i += 1
    doc = nlp(str(abstract)) #transformei o abstract em string, pois tive problemas com abstract sendo float (?)
    for sentence in doc.sents:
        #print(sentence)
        property = re.search("[Yy]ield [Ss]trength", str(sentence))
        #print(property, type(property))
        if property != None:
            #print(property)
            value = re.search("\d+\s[TGMKk]Pa", str(sentence))
            if value != None:
                compositions = re.findall(r"\b[A-Z][a-z]?\d*(?:\.\d+)?(?:[A-Z][a-z]?\d*(?:\.\d+)?){4,8}\b", str(sentence))
                if len(compositions) != 0:
                    print('No Abstract', i, 'a composição:', compositions, 'tem um', property[0], 'de', value[0])
                    print('Sentença de extração do abstract', i, ':', sentence)

Há casos com problemas, por exemplo:

No Abstract 6072 tivemos as seguintes ocorrências:<br>
No Abstract 6073 a composição: ['HfMo0.5NbTiV0.5Si0.7'] tem um yield strength de 1260 MPa<br>
No Abstract 6073 a composição: ['HfMo0.5NbTiV0.5', 'HfMo0.5NbTiV0.5Si0.7'] tem um yield strength de 60 MPa

São duas composições encontradas, temos diferentes valores em MPa também. Vamos conferir o abstract em questão:

In [ ]:
print(dados_total['Abstract'][6072])

Podemos notar alguns problemas na extração, a frase que extraímos foi a seguinte:

"During deformation at 1200 degrees C, the yield strengths of HfMo0.5NbTiV0.5 and HfMo0.5NbTiV0.5Si0.7 alloys are 60 MPa and 235 MPa, respectively."

Haviam dois valores de yield strength, mas o código só pegou 1, porque? Porque usamos o re.search, não o findall. Outro problema é a temperatura que foi de 1200 graus Celcius, não extraímos a temperatura e nem vamos, pois seria muito raro ter todos esses dados em uma sentença, como foi neste caso. Outro ponto foi que as composições HtMo(0.5)NbTiV(0.5) foi ignorada, podemos melhorar o regex para levar isso em conta, mas, por hora não faremos isso.

Vamos corrigir nosso código para encontrar mais que uma ocorrência da unidade que buscamos.

In [ ]:
i = -1
for abstract in dados_total['Abstract']:
    #print('Abstract', i)
    i += 1
    doc = nlp(str(abstract)) #transformei o abstract em string, pois tive problemas com abstract sendo float (?)
    for sentence in doc.sents:
        #print(sentence)
        property = re.search("[Yy]ield [Ss]trength", str(sentence))
        #print(property, type(property))
        if property != None:
            #print(property)
            value = re.findall(r"\d+\s[TGMKk]Pa", str(sentence)) #Alterado para findall
            if len(value) != 0: #Alterado por conta do findall
                compositions = re.findall(r"\b[A-Z][a-z]?\d*(?:\.\d+)?(?:[A-Z][a-z]?\d*(?:\.\d+)?){4,8}\b", str(sentence))
                if len(compositions) != 0:
                    print('No Abstract', i, 'a composição:', compositions, 'tem um', property[0], 'de', value) #Exibindo todos os value agora
                    print('Sentença de extração do abstract', i, ':', sentence)

Agora começamos a notar múltiplos valores para uma composição...

Vamos averiguar alguns dos abstracts em que isso ocorre.

In [ ]:
print(dados_total['Abstract'][146])

Vejam a sentença que continha o yield strength:
    
When the sintering temperature is 1050 degrees C, the A1TiCrNiCu high-entropy alloy has good comprehensive mechanical properties, and its yield strength, compressive strength, plasticity and microhardness are 1410 MPa, 2000 MPa, 9.13% and 5.14 GPa, respectively.

Seria bastante complicado uma regra para extrair isto.

Uma estratégia mais simples, seria apenas considerarmos os casos em que o número de composições seja igual ao número de valores extraídos.

In [ ]:
i = -1
for abstract in dados_total['Abstract']:
    #print('Abstract', i)
    i += 1
    doc = nlp(str(abstract)) #transformei o abstract em string, pois tive problemas com abstract sendo float (?)
    for sentence in doc.sents:
        #print(sentence)
        property = re.search("[Yy]ield [Ss]trength", str(sentence))
        #print(property, type(property))
        if property != None:
            #print(property)
            value = re.findall(r"\s\d*(?:\.\d+)?\s[TGMKk]Pa\b", str(sentence)) #Alterado para findall
            if len(value) != 0: #Alterado por conta do findall
                compositions = re.findall(r"\b[A-Z][a-z]?\d*(?:\.\d*)?(?:[A-Z][a-z]?\d*(?:\.\d+)?){4,8}\b", str(sentence))
                if (len(compositions) != 0) and (len(value) == len(compositions)):
                    print('No Abstract', i, 'a(s) composição(ões):', compositions, 'tem', property[0], 'de', value)
                    print('Sentença de extração do abstract', i, ':', sentence)

Ótimo, vamos então finalizar o código, agora salvando as ocorrências encontradas.

In [ ]:
import pandas as pd
import re
dados_para_df = []

i = -1
for abstract in dados_total['Abstract']:
    i += 1
    doc = nlp(str(abstract))
    for sentence in doc.sents:
        property = re.search("[Yy]ield [Ss]trength", str(sentence))
        if property is not None:
            value = re.findall(r"\s\d*(?:\.\d+)?\s[TGMKk]Pa\b", str(sentence))
            if len(value) != 0:
                compositions = re.findall(r"\b[A-Z][a-z]?\d*(?:\.\d+)?(?:[A-Z][a-z]?\d*(?:\.\d+)?){4,8}\b", str(sentence))
                if len(compositions) != 0 and len(value) == len(compositions):
                    print('No Abstract', i, 'a(s) composição(ões):', compositions, 'tem', property[0], 'de', value)
                    
                    for comp, val in zip(compositions, value):
                        dados_para_df.append({
                            'Composição': comp,
                            'Yield Strength': val,
                            'Sentença': str(sentence)
                        })

dados_extraidos = pd.DataFrame(dados_para_df)

In [ ]:
print(dados_extraidos)

In [ ]:
#!pip install openpyxl
dados_extraidos.to_excel('dados_extraidos.xlsx')

# Playground

In [ ]:
teste1 = "Testando non-capturing group no no no regex"
match1 = re.findall(r"(?:no ){1,3}", teste1)
print(match1)
match2 = re.findall(r"(no ){1,3}", teste1)
print(match2)


In [ ]:
sentence = "Sintering under high pressure at a relatively low temperature was found to contribute to the maintenance of the nanocrystalline state obtained during MA in compact specimens of both AlCoFeCrVNi and AlCoFeCrVTi alloys exhibited extremely high microhardness of 12.6 and 14.7 GPa and yield strength of 3450 and 4100 MPa, respectively."

In [ ]:
value = re.findall(r"\s\d+(?:\.\d+)?\s[TGMKk]Pa\b", str(sentence))
print(value)

# Parte 2 — Extração com LLM (IlumA)

Na Parte 1 chegamos a **45 linhas extraídas de 48.719 sentenças** — e duas delas
estavam erradas de um jeito que nenhum regex resolveria:

> "…the yield strength of V35Ti35Fe15Cr10Zr5 HEA was about **500 MPa higher than**
> that of vanadium alloy…"
> O número é uma **diferença**, não o valor da propriedade.

> "…maximum **compressive** yield strength of 1479 MPa"
> É o limite **em compressão** — outra grandeza.

Nos dois casos a sintaxe está certa e o **sentido** está errado. Regex casa forma;
não entende frase. É exatamente essa lacuna que um modelo de linguagem preenche.

Agora vamos refazer **a mesma extração**, nas mesmas frases, usando a IlumA — o
servidor de LLM da Ilum — e comparar os dois resultados.

## Tutorial: obtendo acesso à IlumA

A IlumA é um *gateway* de LLM: um endereço único que fala o mesmo protocolo da API
da OpenAI. Você precisa de um **token** pessoal para usá-la.

### Passo 1 — Entre com a conta da Ilum

Acesse **https://iluma.cnpem.br:4000/** e clique em **Sign in** (canto superior
direito). Na tela de login, escolha **"Continue with Microsoft"** e use a sua conta
institucional.

![Tela de login da New API](imagens_iluma/iluma_1_login.jpg)

### Passo 2 — Abra o Console

Depois de entrar, clique em **Console** na barra de cima. É o painel da sua conta:
consumo, requisições e o menu lateral com tudo o que você vai usar.

No menu à esquerda, clique em **Token Management**.

![Console da New API](imagens_iluma/iluma_2_console.jpg)

### Passo 3 — Crie o token

Clique em **Create token** e preencha:

* **Name** — um nome que você reconheça depois, por exemplo `meu-token-aula04`.
* **Token Group** — deixe o padrão (o seu grupo).
* **Expiration time** — `Never expires` serve para a disciplina. Em trabalho de
  verdade, prefira um prazo.
* **New quantity** — 1.
* **Quota** — deixe *Unlimited quota* ligado; quem limita de fato é a cota da conta.

Depois, **Submit**.

![Criação de token](imagens_iluma/iluma_3_criar_token.jpg)

### Passo 4 — Copie a chave

O token aparece na lista, com a chave escondida no meio (`sk-OAb8**********95Rj`).
O ícone do **olho** revela e o de **cópia** copia a chave inteira.

![Token criado](imagens_iluma/iluma_4_token_criado.jpg)

> ⚠️ **O token é uma senha.** Ele gasta a cota da sua conta e não deve aparecer em
> nenhum lugar público. Não escreva a chave dentro de uma célula do notebook, não
> mande por chat e não suba para o GitHub — é o erro mais comum de quem está
> começando. Nas células abaixo ela é digitada com `getpass`, que não deixa rastro
> no arquivo salvo.

## Conectando

A IlumA fala o protocolo da OpenAI, então usamos a biblioteca `openai` apontando
para outro endereço. Só duas coisas mudam em relação a um exemplo da OpenAI:
a `base_url` e o nome do modelo.

In [ ]:
!pip install openai

In [ ]:
import getpass
import json
import random
import re          # a nativa: a Parte 1 usou `regex`, aqui a padrão basta
import time

import pandas as pd
from openai import OpenAI

BASE_URL = "https://iluma.cnpem.br:4000/v1"
MODELO = "iluma"          # o único modelo publicado no gateway

# O manual de qualquer API diz "para extração, use temperature=0". Aqui não dá:
# o modelo por trás da IlumA é um Qwen e, nesta instalação, valores abaixo de
# ~0,5 fazem a chamada TRAVAR — ela simplesmente não retorna. 0,5 é o mais
# determinístico que conseguimos.
TEMP_EXTRACAO = 0.5

# getpass não ecoa o que você digita e não guarda nada no notebook salvo.
token = getpass.getpass("Cole o seu token da IlumA (sk-...): ")

# timeout explícito: sem ele, uma chamada que trava fica pendurada para sempre e
# só sai com Ctrl-C. Melhor falhar em 60 s e saber que falhou.
cliente = OpenAI(base_url=BASE_URL, api_key=token, timeout=60.0, max_retries=1)

In [ ]:
# Teste de conexão — se isto responder, está tudo certo.
resposta = cliente.chat.completions.create(
    model=MODELO,
    messages=[{"role": "user", "content": "Responda apenas: conexão OK."}],
)
print(resposta.choices[0].message.content)

## System Prompt e User Prompt

Uma conversa com o modelo é uma **lista de mensagens**, e cada uma tem um papel:

* **System prompt** — define *quem o modelo é e quais regras ele segue*. Vem antes
  de tudo e vale para a conversa inteira: o papel ("você é um extrator de dados de
  artigos de materiais"), o que é permitido, o formato da resposta. É onde se coloca
  aquilo que não deve mudar a cada pergunta.
* **User prompt** — é *o pedido específico de agora*. No nosso caso, a sentença da
  qual queremos extrair os dados.

A separação importa por uma razão prática: o system prompt é escrito **uma vez** e
reaproveitado em milhares de chamadas, enquanto o user prompt muda a cada item. É a
mesma ideia de separar a função dos seus argumentos.

In [ ]:
def perguntar(pergunta, sistema=None, **kw):
    """Uma chamada ao modelo. `sistema` é o system prompt; `pergunta`, o do usuário."""
    mensagens = []
    if sistema:
        mensagens.append({"role": "system", "content": sistema})
    mensagens.append({"role": "user", "content": pergunta})
    r = cliente.chat.completions.create(model=MODELO, messages=mensagens, **kw)
    return r.choices[0].message.content


# A mesma pergunta, com e sem system prompt:
pergunta = "O que é uma liga de alta entropia?"

print("SEM system prompt:")
print(perguntar(pergunta)[:400], "...\n")

print("COM system prompt:")
print(perguntar(
    pergunta,
    sistema="Você responde a estudantes de graduação em uma frase, sem jargão.",
))

## O prompt de assistente

O terceiro papel é o **`assistant`**: as mensagens do próprio modelo.

Para entender por que ele existe, é preciso engolir um fato que costuma
surpreender: **o modelo não tem memória.** Cada chamada à API é completamente
independente da anterior. Ele não sabe que você existe, não sabe o que perguntou
há dez segundos, e entre uma chamada e outra não guarda absolutamente nada.

Então como um chat "lembra" da conversa? **Você reenvia tudo, toda vez.** A cada
nova pergunta, o programa monta de novo a lista inteira — system, a primeira
pergunta, a primeira resposta, a segunda pergunta — e manda o pacote completo. As
respostas anteriores entram nessa lista como mensagens de papel `assistant`. A
memória não está no modelo: está na lista que você carrega.

A célula abaixo mostra isso sem meias palavras.

In [ ]:
# Primeiro turno, normal.
p1 = "Qual é a fórmula da liga Cantor?"
r1 = perguntar(p1)
print("P:", p1)
print("R:", r1[:200], "...\n")

# Segundo turno SEM histórico: "ela" não se refere a nada.
p2 = "Quantos elementos ela tem?"
print("--- sem histórico")
print("R:", perguntar(p2)[:200], "...\n")

# Segundo turno COM histórico: a resposta anterior volta como `assistant`.
print("--- com histórico (o papel assistant)")
r = cliente.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "user", "content": p1},
        {"role": "assistant", "content": r1},   # ← o que ELE respondeu antes
        {"role": "user", "content": p2},
    ],
)
print("R:", r.choices[0].message.content[:200], "...")

### O truque: mensagens `assistant` que nunca aconteceram

Aqui está a parte que quase ninguém percebe na primeira vez. Nada obriga a mensagem
`assistant` a ser algo que o modelo realmente disse. **Você pode escrevê-la.**

E é isso que faz dela uma ferramenta, não só um registro. Se você monta um par —
uma mensagem `user` com uma entrada de exemplo e uma mensagem `assistant` com a
resposta ideal para aquela entrada — o modelo lê aquilo como "foi assim que eu
respondi da última vez" e passa a imitar o padrão.

Isso se chama ***few-shot*** (contra *zero-shot*, quando você só descreve o que
quer). Na prática:

* **descrever** o formato = escrever um parágrafo no system prompt;
* **mostrar** o formato = um par `user`/`assistant` de exemplo.

Mostrar quase sempre funciona melhor, e é muito mais curto. Vale principalmente
quando a saída tem forma rígida — como o JSON que vamos pedir daqui a pouco. Um
exemplo bem escolhido resolve de uma vez a vírgula, as aspas, o nome dos campos e o
que fazer quando não há nada a extrair.

In [ ]:
# few-shot: o par abaixo NUNCA aconteceu — foi escrito por nós.
# Repare que o formato do JSON não é descrito em lugar nenhum. Só é mostrado uma
# vez, e isso basta para a resposta seguinte sair no mesmo molde.
resposta = cliente.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Você extrai dados de frases sobre ligas."},
        {"role": "user", "content": "The alloy CoCrFeNi showed a yield strength of 250 MPa."},
        {"role": "assistant", "content": '{"composicao": "CoCrFeNi", "yield_MPa": 250}'},
        {"role": "user", "content": "Al0.5CoCrFeNi reached 1.2 GPa of yield strength."},
    ],
    temperature=TEMP_EXTRACAO,
)
print(resposta.choices[0].message.content)

## Os três parâmetros de amostragem

O modelo não escolhe a próxima palavra: ele calcula uma **probabilidade para cada
palavra possível** e sorteia. Estes três parâmetros controlam esse sorteio.

* **`temperature`** — o quanto o sorteio é ousado. Perto de `0`, o modelo escolhe
  quase sempre a palavra mais provável (respostas repetíveis, boas para extração).
  Acima de `1`, ele arrisca mais (bom para texto criativo, ruim para dados).
* **`top_p`** — em vez de considerar todas as palavras, considera só as mais
  prováveis até que a soma das probabilidades chegue a `p` (ex.: `0.9`) e sorteia
  entre elas. Corta a cauda de opções improváveis.
* **`top_k`** — a mesma ideia, mas por contagem fixa: só as `k` palavras mais
  prováveis entram no sorteio.

A receita de manual para extração de dados é **`temperature=0`**: você quer o mesmo
resultado se rodar duas vezes.

### Só que aqui não dá

O modelo por trás da IlumA é um **Qwen**, e nesta instalação **valores abaixo de
~0,5 travam a chamada** — ela não retorna nem dá erro; fica pendurada até você
interromper. Por isso o nosso "o mais determinístico possível" é
`TEMP_EXTRACAO = 0.5`.

Guarde a lição, que vale para qualquer API: **o que o manual recomenda e o que a
instalação aceita nem sempre coincidem.** Isso não está escrito em documentação
nenhuma — descobre-se testando, e é por isso que a primeira coisa a fazer com uma
API nova é justamente sondar os limites dela.

**Voltaremos a esses parâmetros com calma nas próximas aulas** — por ora basta saber
que existem e por que aqui usamos 0,5.

In [ ]:
# A mesma pergunta, três vezes, com duas temperaturas.
# ATENÇÃO: não baixe de 0,5 — abaixo disso esta instalação trava.
for t in (TEMP_EXTRACAO, 1.3):
    print(f"--- temperature={t}")
    for _ in range(3):
        print("   ", perguntar("Cite UMA liga de alta entropia. Responda só o nome.",
                               temperature=t))

## A mesma extração da Parte 1, agora com o modelo

O system prompt abaixo é o coração do exercício. Note que ele não descreve *padrões
de texto* — descreve **o que conta como um dado válido**. As duas regras que salvam os
casos que o regex errou estão ali, escritas em português:

* um número que aparece como *diferença* ("X MPa higher than") não é um valor;
* se o ensaio for de compressão, isso precisa ser registrado.

E exigimos **JSON**, para a resposta entrar direto num `DataFrame`.

In [ ]:
SISTEMA_EXTRACAO = """Você extrai dados de resumos de artigos sobre ligas metálicas.

Da sentença fornecida pelo usuário, extraia os pares (composição da liga, limite de
escoamento). Regras:

1. Extraia APENAS o limite de escoamento (yield strength). Ignore dureza (hardness,
   HV), limite de resistência à tração (tensile/compressive strength), alongamento
   (elongation) e qualquer outra propriedade.
2. Se o número for uma DIFERENÇA ou COMPARAÇÃO ("500 MPa higher than", "increased by
   200 MPa"), NÃO é um valor da propriedade: ignore.
3. Se o ensaio for de compressão, registre "compressao" em `condicao`; se for de
   tração ou não estiver dito, use "tracao" ou "nao_informado".
4. Se a temperatura do ensaio estiver na sentença, registre em `temperatura`.
5. Converta o valor para MPa (1 GPa = 1000 MPa).

Responda APENAS com JSON válido, sem cercas de markdown, neste formato:

{"extracoes": [{"composicao": "AlCoCrFeNi", "yield_strength_MPa": 1511.0,
                "condicao": "tracao", "temperatura": null}]}

Se não houver nada a extrair, responda {"extracoes": []}."""


def limpar_cercas(texto: str) -> str:
    """Alguns modelos embrulham o JSON em ```json ... ``` mesmo quando pedimos que não."""
    t = texto.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t)
    return t


def extrair_llm(sentenca: str) -> list[dict]:
    bruto = perguntar(sentenca, sistema=SISTEMA_EXTRACAO, temperature=TEMP_EXTRACAO)
    try:
        return json.loads(limpar_cercas(bruto)).get("extracoes", [])
    except json.JSONDecodeError:
        print("  (resposta não era JSON):", bruto[:120])
        return []

In [ ]:
# A sentença com que a aula começou:
sentenca = ("The yield strength of the AlCuNiFeCr alloy reached 1511 MPa "
            "at room temperature.")
extrair_llm(sentenca)

### Os dois casos em que o regex errou

Aqui está o teste que interessa. São as duas linhas que a revisão manual precisou
tirar da planilha da Parte 1. O regex extraiu as duas com toda a confiança.

In [ ]:
casos = [
    # o número é uma diferença, não o valor
    "At 700 degrees C, the yield strength of V35Ti35Fe15Cr10Zr5 HEA was about "
    "500 MPa higher than that of vanadium alloy or martensitic steel.",
    # o valor é de compressão
    "Zr20Ti20Nb20Mo20Cr20 HEA shows the maximum Vickers hardness of 564 HV and "
    "maximum compressive yield strength of 1479 MPa, respectively.",
    # e o caso que a Parte 1 declarou difícil demais para uma regra
    "When the sintering temperature is 1050 degrees C, the A1TiCrNiCu high-entropy "
    "alloy has good comprehensive mechanical properties, and its yield strength, "
    "compressive strength, plasticity and microhardness are 1410 MPa, 2000 MPa, "
    "9.13% and 5.14 GPa, respectively.",
]

for i, c in enumerate(casos, 1):
    print(f"--- caso {i}")
    print("   ", c[:90], "...")
    print("   →", extrair_llm(c))
    time.sleep(0.3)

## Em escala: o modelo contra o regex, nas mesmas frases

Vamos separar uma amostra de sentenças que mencionam *yield strength* e rodar as duas
abordagens em cada uma.

⚠️ **Cada chamada consome cota da sua conta.** Comece com poucas sentenças. Rodar as
1.001 do corpus inteiro custa dinheiro real e leva bastante tempo — se quiser fazer
isso, faça com consciência do custo.

In [ ]:
ALVO = re.compile(r"[Yy]ield [Ss]trength")

# Reaproveita `dados_total` e `nlp`, já carregados na Parte 1.
sentencas = []
for abstract in dados_total['Abstract'].dropna().astype(str)[:400]:
    for s in nlp(abstract).sents:
        if ALVO.search(str(s)):
            sentencas.append(str(s))

print(len(sentencas), "sentenças com 'yield strength' nos 400 primeiros resumos")
amostra = random.Random(42).sample(sentencas, min(15, len(sentencas)))

In [ ]:
# O pipeline de regex da Parte 1, embrulhado numa função para poder comparar.
P_VALOR = re.compile(r"\s\d*(?:\.\d+)?\s[TGMKk]Pa\b")
P_COMP = re.compile(r"\b[A-Z][a-z]?\d*(?:\.\d+)?"
                    r"(?:[A-Z][a-z]?\d*(?:\.\d+)?){4,8}\b")


def extrair_regex(sentenca: str) -> list[tuple[str, str]]:
    valores = P_VALOR.findall(sentenca)
    comps = P_COMP.findall(sentenca)
    if comps and len(valores) == len(comps):
        return list(zip(comps, [v.strip() for v in valores]))
    return []

In [ ]:
linhas = []
for i, s in enumerate(amostra, 1):
    print(f"[{i}/{len(amostra)}]", end=" ", flush=True)
    por_regex = extrair_regex(s)
    por_llm = extrair_llm(s)
    linhas.append({
        "sentenca": s,
        "regex_n": len(por_regex),
        "llm_n": len(por_llm),
        "regex": por_regex,
        "llm": por_llm,
    })
    time.sleep(0.3)

comparacao = pd.DataFrame(linhas)
print("\n\nsentenças em que o regex achou algo:", (comparacao.regex_n > 0).sum())
print("sentenças em que o LLM achou algo:  ", (comparacao.llm_n > 0).sum())
comparacao[["regex_n", "llm_n", "sentenca"]]

In [ ]:
# Olhe caso a caso: onde os dois discordam é onde está o aprendizado.
for _, r in comparacao.iterrows():
    if r.regex_n != r.llm_n:
        print("SENTENÇA:", r.sentenca[:150], "...")
        print("  regex:", r.regex)
        print("  llm  :", r.llm)
        print()

## O que mudou, e o que não mudou

O modelo cobre muito mais frases e entende o que o regex não entendia — a diferença,
a compressão, a lista com "respectively". Mas ele traz problemas novos, e nenhum
deles é pequeno:

* **Custa dinheiro e tempo.** Um regex roda 48 mil sentenças em segundos, de graça.
* **Não é determinístico por natureza.** E aqui nem sequer podemos zerar a
  temperatura: 0,5 é o piso desta instalação, então rodar duas vezes pode dar
  respostas diferentes. Um regex dá sempre a mesma.
* **Pode inventar.** Um regex nunca devolve um número que não está no texto; um
  modelo pode. Isso se chama *alucinação*, e é por isso que a validação por amostra
  continua obrigatória — exatamente como na Parte 1.
* **Manda o seu texto para um servidor.** Aqui é o da Ilum, mas o hábito importa:
  dado sigiloso não sai da sua máquina sem você decidir isso conscientemente.

Regra prática: **regex para o que tem forma fixa** (unidades, DOIs, códigos, datas),
**LLM para o que depende do sentido da frase**. Quase todo pipeline sério usa os dois.

## Exercícios — Parte 2

1. Reescreva o `SISTEMA_EXTRACAO` para extrair **dureza** (Vickers, HV) em vez do
   limite de escoamento. Quantas regras você precisou mudar?
2. Rode a mesma sentença cinco vezes com `temperature=0.5` e cinco com
   `temperature=1.2`. As respostas mudam? Qual você usaria num pipeline de dados?
   (Não tente `0`: nesta instalação a chamada trava.)
3. Pegue as 15 sentenças da amostra e **confira à mão** as extrações do modelo.
   Calcule a precisão e compare com a do regex, que você mediu no exercício 6 da
   Parte 1. Mesma amostra, mesma métrica.
4. Encontre uma sentença em que o modelo **inventa** um valor que não está no texto.
   (Dica: frases com muitos números e várias propriedades.)
5. Tire a regra 2 do system prompt (a da diferença) e rode o caso 1 de novo. O que
   acontece? O que isso diz sobre onde mora a inteligência da solução?
6. Estime o custo: veja em **Console → Usage Logs** quantos tokens as suas chamadas
   consumiram e projete quanto custaria rodar as 1.001 sentenças do corpus.